# Langfuse Demo — Chapter 1: Setup and First Trace

> **Prerequisite**: `docker compose up -d` running

---

## Why observability for LLMs?

You put a model in production. Then the questions start:

- Is it answering correctly? How often does it fail?
- What is the average latency? And p95?
- How many tokens is it consuming per call? What is the cost?
- Which questions can't the model answer well?
- When I updated the prompt, did quality improve or worsen?

**Langfuse answers all these questions**, without changing your business logic.

---

## Architecture used in this handover

```
Notebook / Python Code
        │
        │  Langfuse SDK (traces, spans, scores)
        ▼
  Langfuse (self-hosted, port 3000)
        │
        ├── PostgreSQL  (metadata)
        ├── ClickHouse  (analytics)
        └── MinIO       (artifacts)
```

Groq API (external LLM) → we use llama-3.3-70b on the free tier

## 1. Connect to Langfuse

The SDK reads keys from `.env` automatically.
Make sure the `.env` file exists (copy from `.env.example`) and Docker is running.

In [1]:
from langfuse_demo.client import get_client
from langfuse_demo.config import settings
from langfuse_demo.llm import call_llm
from dotenv import load_dotenv

_ = load_dotenv()

langfuse = get_client()

if langfuse.auth_check():
    print(f"Connected to Langfuse!")
    print(f"UI available at: {settings.langfuse_host}")
else:
    print("Authentication failed.")
    print("Check: docker compose ps | .env (LANGFUSE_PUBLIC_KEY / SECRET_KEY)")

Connected to Langfuse!
UI available at: http://localhost:3000


---

## 2. Your first trace — 2 lines of code

The `@observe` decorator instruments any function **without modifying its logic**.
Automatically captures: name, inputs, outputs, latency, and errors.

```python
from langfuse import observe

@observe()          # ← line 1
def my_function(question: str) -> str:
    answer, _ = call_llm([{"role": "user", "content": question}])
    return answer  # ← line 2 (already in the trace!)
```

Let's see this live:

In [2]:
from langfuse import observe

@observe(name="answer-question")          # automatically creates a Trace
def answer(question: str) -> str:
    response, _ = call_llm([{"role": "user", "content": question}])
    return response

response = answer("What is a Large Language Model, in 2 sentences?")
print(response)

langfuse.flush()  # ensures the trace reached the server
print(f"\nView in UI: {settings.langfuse_host}/traces")

A Large Language Model (LLM) is a type of artificial intelligence (AI) designed to process and understand human language, using complex algorithms and massive datasets to generate text, answer questions, and complete tasks. LLMs are trained on vast amounts of text data, allowing them to learn patterns, relationships, and context, and can be fine-tuned for specific applications such as language translation, text summarization, and conversation generation.

View in UI: http://localhost:3000/traces


**What to open in the UI now:**
- Go to **Traces** → you will see `answer-question`
- Click the trace → see: input, output, latency
- `@observe` captured everything with no extra lines of code

---

## 3. Trace with rich metadata

Day to day, you want more context: which user asked the question? In what environment?  
Add metadata, user_id, tags:

In [3]:
from langfuse import observe, propagate_attributes

@observe(name="docs-assistant")
def assistant(question: str, user_id: str, session_id: str) -> str:
    # propagate_attributes adds context to the entire trace
    with propagate_attributes(
        user_id=user_id,
        session_id=session_id,
        tags=["demo", "ch-01"],
        metadata={"environment": "ds-handover"},
    ):
        messages = [
            {"role": "system", "content": "You are a technical assistant. Answer clearly and concisely."},
            {"role": "user", "content": question},
        ]
        response, usage = call_llm(messages)
        print(f"Tokens — input: {usage['input']} | output: {usage['output']}")
        return response

response = assistant(
    question="What is the difference between precision and recall in classification?",
    user_id="marcos.pereira",
    session_id="langfuse-handover-2024",
)
print(f"\nAnswer:\n{response}")

langfuse.flush()
print(f"\nView traces: {settings.langfuse_host}/traces")

Tokens — input: 59 | output: 278

Answer:
In classification, precision and recall are two important metrics used to evaluate the performance of a model.

* **Precision**: It measures the number of true positives (correctly predicted instances) out of all the instances predicted as positive by the model. In other words, it measures the accuracy of the model's positive predictions. 
  Formula: Precision = True Positives / (True Positives + False Positives)

* **Recall**: It measures the number of true positives out of all the actual positive instances in the dataset. In other words, it measures the model's ability to detect all the positive instances. 
  Formula: Recall = True Positives / (True Positives + False Negatives)

To illustrate the difference:

Example: 
- True Positives: 80 (correctly predicted positive instances)
- False Positives: 20 (incorrectly predicted positive instances)
- False Negatives: 30 (missed positive instances)

Precision = 80 / (80 + 20) = 0.8 (80% of predicte

---

## 4. Manual tracing — full control

When you need to capture **tokens, model, and cost** explicitly,  
use `start_as_current_observation` with `as_type="generation"`.

This allows you to:
- Record input/output tokens for cost calculation
- Specify the model used
- Get the `trace_id` to create scores later

In [4]:
QUESTION = "Explain the concept of overfitting in 3 lines."

messages = [
    {"role": "system", "content": "You are an ML expert. Answer clearly and concisely."},
    {"role": "user", "content": QUESTION},
]

with langfuse.start_as_current_observation(
    as_type="generation",
    name="explain-concept",
    input=messages,
    metadata={"topic": "overfitting", "demo": "ch-01"},
) as gen:
    response, usage = call_llm(messages)
    gen.update(
        model=settings.groq_model,
        output=response,
        usage_details={"input": usage["input"], "output": usage["output"]},
        cost_details={"input": usage["input_cost"], "output": usage["output_cost"]},
    )
    trace_id = langfuse.get_current_trace_id()

langfuse.flush()

print(f"Answer: {response}")
print(f"\nTokens — input: {usage['input']} | output: {usage['output']}")
print(f"Cost — input: ${usage['input_cost']:.6f} | output: ${usage['output_cost']:.6f}")
print(f"Trace ID : {trace_id}")
print(f"Direct link: {settings.langfuse_host}/trace/{trace_id}")

Answer: Overfitting occurs when a model is too complex and learns the noise in the training data. As a result, it performs well on the training set but poorly on new, unseen data. This happens when the model is overly specialized to the training data, rather than generalizing to the underlying patterns.

Tokens   — input: 61 | output: 61
Cost     — input: $0.000036 | output: $0.000048
Trace ID : 257a9695f4ea1dc4c071af6b74cb554b
🔗 Direct link: http://localhost:3000/trace/257a9695f4ea1dc4c071af6b74cb554b


---

## Chapter 1 Summary

| What we saw | How to do it |
|-------------|------------|
| Automatic trace | `@observe()` on any function |
| Context in trace | `propagate_attributes(user_id=..., tags=...)` |
| Tokens + cost | `start_as_current_observation(as_type="generation")` + `gen.update(usage_details=...)` |
| Direct link to trace | `langfuse.get_current_trace_id()` |

**Next chapter**: Complete RAG pipeline with span hierarchy → retrieval → reranking → generation